# Dataset description
## Kidnapping rate per 100k inhabitants 

**Kidnapping cases**: Ministerio de Defensa Nacional, [download](https://www.datos.gov.co/Seguridad-y-Defensa/SECUESTRO/d7zw-hpf4/about_data)

**Population**: Dane, [download](https://www.dane.gov.co/index.php/estadisticas-por-tema/demografia-y-poblacion/proyecciones-de-poblacion)

# Code
## Imports

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from pathlib import Path
import matplotlib.cm as cm
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize
from highlight_text import fig_text
import unicodedata
import json

BASE_DIR = Path.cwd().parent.parent

## Load data

In [22]:
dep_path = "./data/departamental.csv"
kid_path = "./data/SECUESTRO_20260804.csv"
pop_path = "./data/nacional.csv"
path_geo = "./data/colombia_departments.geojson"
logo_path = BASE_DIR / "logo/logo.png"

dep = pd.read_csv(dep_path, encoding="utf-8", usecols=["DPNOM", "AÑO", "Total"])
kid = pd.read_csv(kid_path, encoding="utf-8", usecols=["FECHA HECHO", "DEPARTAMENTO", "CANTIDAD"])
pop = pd.read_csv(pop_path, encoding="utf-8", usecols=["AÑO", "Total"]) 
logo = plt.imread(logo_path)

with open(path_geo, encoding="utf-8") as f:
    geo = json.load(f)

## Clean data

In [23]:
kid["FECHA HECHO"] = pd.to_datetime(kid["FECHA HECHO"], dayfirst=True)
kid["FECHA HECHO"] = kid["FECHA HECHO"].dt.year
total = kid.groupby(["FECHA HECHO", "DEPARTAMENTO"]).sum().reset_index()


def normalize_name(name: str) -> str:
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    ascii_name = ascii_name.upper().replace(".", "").replace(",", "")
    return " ".join(ascii_name.split())


# Each source spells departments differently (accents, "D.C." suffixes, San
# Andrés long-form name), so map every name to the geojson's DPTO code and
# merge on that instead of raw strings.
geo_name_to_dpto = {
    normalize_name(feat["properties"]["NOMBRE_DPT"]): feat["properties"]["DPTO"]
    for feat in geo["features"]
}
geo_name_to_dpto.update({
    "BOGOTA DC": "11",
    "ARCHIPIELAGO DE SAN ANDRES": "88",
    "SAN ANDRES ISLAS": "88",
})

total["DPTO"] = total["DEPARTAMENTO"].map(lambda s: geo_name_to_dpto[normalize_name(s)])
dep["DPTO"] = dep["DPNOM"].map(lambda s: geo_name_to_dpto[normalize_name(s)])

merged = pd.merge(
    total,
    dep,
    left_on=["FECHA HECHO", "DPTO"],
    right_on=["AÑO", "DPTO"],
    how="left"
)

merged["ratio"] = (merged["CANTIDAD"] / merged["Total"]) * 100000

total = merged.groupby("FECHA HECHO")[["CANTIDAD"]].sum().reset_index()
total = pd.merge(total, pop, left_on="FECHA HECHO", right_on="AÑO", how="inner")
total["ratio"] = (total["CANTIDAD"] / total["Total"]) * 100000

##  Plot

In [24]:
# 2026 is a partial year (data pulled 2026-08-04), so it's excluded.
years = range(2003, 2026)

dept_rates = {
    year: merged.loc[merged["FECHA HECHO"] == year].set_index("DPTO")["ratio"]
    for year in years
}
national_rates = {
    year: total.loc[total["FECHA HECHO"] == year, "ratio"].iloc[0]
    for year in years
}

# Shared scale so every year's map is directly comparable.
vmax = max(s.max() for s in dept_rates.values())

In [25]:
missing_color = "#D3D3D3"
cmap = cm.Reds
norm = Normalize(vmin=0, vmax=vmax)
background = "#F4F1EA"

sanandres_box = {"lon": (-79.0, -75.9), "lat": (10.9, 13.3)}


def exterior_rings(geometry: dict) -> list:
    if geometry["type"] == "Polygon":
        return [geometry["coordinates"][0]]
    return [poly[0] for poly in geometry["coordinates"]]


def fit_box(rings: list, box: dict) -> list:
    """Uniform scale + center rings into a target lon/lat box (preserve aspect)."""
    xs = [x for r in rings for x, _ in r]
    ys = [y for r in rings for _, y in r]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    bx0, bx1 = box["lon"]
    by0, by1 = box["lat"]
    scale = min((bx1 - bx0) / (x1 - x0), (by1 - by0) / (y1 - y0))
    ox = bx0 + ((bx1 - bx0) - (x1 - x0) * scale) / 2
    oy = by0 + ((by1 - by0) - (y1 - y0) * scale) / 2
    return [[(ox + (x - x0) * scale, oy + (y - y0) * scale) for x, y in r]
            for r in rings]


def plot_rate_map(rates: pd.Series, national: float, year: int):
    fig, ax = plt.subplots(figsize=(10.8, 10), dpi=500)
    fig.patch.set_facecolor(background)
    ax.patch.set_facecolor(background)

    patches, colors = [], []
    for feat in geo["features"]:
        code = feat["properties"]["DPTO"]
        val = rates.get(code)
        color = missing_color if pd.isna(val) else cmap(norm(val))
        rings = exterior_rings(feat["geometry"])
        if code == "88":
            def bbox_area(r):
                xs = [x for x, _ in r]; ys = [y for _, y in r]
                return (max(xs) - min(xs)) * (max(ys) - min(ys))
            main = max(rings, key=bbox_area)
            rings = fit_box([main], sanandres_box)
        for ring in rings:
            patches.append(Polygon(ring, closed=True))
            colors.append(color)

    ax.add_collection(PatchCollection(
        patches, facecolor=colors, edgecolor="white", linewidth=0.4))
    ax.autoscale_view()
    ax.set_aspect("equal")
    ax.axis("off")
    # Left-anchor the map so it fills the empty space on the left.
    ax.set_position([0.02, 0.05, 0.78, 0.85])

    ax.text(-77.45, 10.6, "San Andrés", fontsize=8, color="gray",
            ha="center", va="top")

    fig_text(
        x=0.125, y=0.95,
        s=f"Kidnapping rate by department · <{year}>",
        highlight_textprops=[{"color": cmap(0.85), "weight": "bold"}],
        fontsize=15,
    )

    fig_text(
        x=0.125, y=0.915,
        s="per 100,000 inhabitants",
        fontsize=10, color="gray",
    )

    # National rate metric.
    fig_text(
        x=0.72, y=0.90,
        s=f"<{national:.1f}>\nNational rate · per 100k",
        highlight_textprops=[{"color": cmap(0.85), "weight": "bold", "fontsize": 28}],
        fontsize=10, color="gray",
    )

    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    cax = fig.add_axes([0.84, 0.3, 0.02, 0.4])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Kidnappings per 100,000", fontsize=8, color="gray")
    cbar.ax.tick_params(labelsize=8, colors="gray")
    cbar.outline.set_visible(False)

    fig.text(
        0.125, 0.05,
        "Departments in grey have no data for the selected year\n"
        "Source: Ministerio de Defensa Nacional, DANE.",
        fontsize=8, color="gray")

    imagebox = OffsetImage(logo, zoom=0.4)
    ab = AnnotationBbox(imagebox, xy=(0.7, 0.03), xycoords="figure fraction",
                         frameon=False, pad=0)
    fig.add_artist(ab)

    fig.savefig(f"/wherever/you/want/{year}.png",
                dpi=500, bbox_inches="tight")
    plt.close(fig)


for year in years:
    plot_rate_map(dept_rates[year], national_rates[year], year)